# Training Pipeline — LogBERT + DeepSVDD

Deteksi anomali dataset BGL. Tuned untuk **RTX 4050 (6 GB VRAM)**.

Notebook ini menjalankan Tahap 1–6 secara bertahap agar bisa diverifikasi per-cell.
Parameter diimpor dari `scripts/training/config.py` (satu sumber kebenaran).

**Verifikasi dulu:** set `LIMIT` dan `SMOKE` di cell Setup, jalankan semua cell.
Kalau lancar tanpa error, set `LIMIT=None` & `SMOKE=False` untuk training penuh.

## Setup — import config & cek GPU

In [ ]:
import sys, pickle
from pathlib import Path

# Tambahkan root project + folder scripts/training ke path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))                          # agar bisa 'from src.model...'
sys.path.insert(0, str(ROOT / 'scripts' / 'training')) # agar bisa 'import config'
import config as C

import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

# ── Mode verifikasi ──────────────────────────────────────────────
LIMIT = 50000   # None untuk seluruh dataset; angka kecil untuk verifikasi cepat
SMOKE = True    # True = training singkat (200 sampel, 1 epoch) untuk cek program
print(f'LIMIT={LIMIT} | SMOKE={SMOKE}')

## Tahap 1 — Muat BGL.log

Butuh `data/BGL.log` (download dari LOGHUB, folder BGL).

In [ ]:
import pandas as pd

def load_bgl(path, limit=None):
    rows = []
    with open(path, encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            parts = line.strip().split(None, 8)
            if len(parts) < 9:
                continue
            rows.append({
                'label': parts[0], 'is_anomaly': parts[0] != '-',
                'timestamp_unix': int(parts[1]), 'node': parts[3],
                'component': parts[6], 'level': parts[7], 'content': parts[8],
            })
    df = pd.DataFrame(rows)
    df['datetime'] = pd.to_datetime(df['timestamp_unix'], unit='s')
    return df.sort_values('datetime').reset_index(drop=True)

assert C.RAW_LOG.exists(), f'{C.RAW_LOG} tidak ada. Download BGL.log dari LOGHUB.'
df = load_bgl(C.RAW_LOG, LIMIT)
df.to_pickle(C.RAW_DF)
print(f'{len(df):,} baris | anomali {df["is_anomaly"].mean():.2%}')
df.head()

### Eksplorasi cepat — distribusi label

In [ ]:
import matplotlib.pyplot as plt

counts = df['is_anomaly'].value_counts()
counts.plot(kind='bar', color=['#4c9', '#e55'])
plt.title('Distribusi Normal vs Anomali (level baris)')
plt.xticks([0, 1], ['Normal', 'Anomali'], rotation=0)
plt.ylabel('Jumlah baris')
plt.show()

## Tahap 2 — Parsing Drain3 → Log Key

State parser disimpan ke `models/drain3_state.bin` (WAJIB dipakai ulang saat inference).

In [ ]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from drain3.file_persistence import FilePersistence

cfg = TemplateMinerConfig(); cfg.drain_sim_th = 0.4; cfg.drain_depth = 4
miner = TemplateMiner(persistence_handler=FilePersistence(str(C.DRAIN_STATE)), config=cfg)

log_keys = []
for i, content in enumerate(df['content']):
    log_keys.append(miner.add_log_message(content)['cluster_id'])
    if (i + 1) % 200_000 == 0:
        print(f'  {i+1:,} diproses ...')

df['log_key'] = log_keys
df.to_pickle(C.PARSED_DF)
miner.save_state('training complete')
print(f'{len(miner.drain.clusters)} template unik | state → {C.DRAIN_STATE.name}')

## Tahap 3 — Sliding window 5 menit

In [ ]:
# Sliding window 5 menit — vektorized via bucket Unix timestamp (cepat untuk dataset penuh)
sec = C.WINDOW_MINUTES * 60
df = df.sort_values('timestamp_unix').reset_index(drop=True)
df['win_idx'] = df['timestamp_unix'] // sec        # tiap baris → bucket 5 menit

windows = []
for wid, (_, chunk) in enumerate(df.groupby('win_idx', sort=True)):
    windows.append({
        'window_id': f'win_{wid:06d}',
        'timestamp': chunk['datetime'].iloc[0].isoformat(),
        'log_keys': chunk['log_key'].astype(str).tolist(),
        'log_sequence': chunk['content'].tolist(),
        'is_anomaly': bool(chunk['is_anomaly'].any()),
    })

pickle.dump(windows, open(C.WINDOWS_PKL, 'wb'))
n_anom = sum(w['is_anomaly'] for w in windows)
avg_len = sum(len(w['log_keys']) for w in windows) / max(len(windows), 1)
print(f'{len(windows):,} window | anomali {n_anom:,} ({n_anom/max(len(windows),1):.2%}) | rata2 {avg_len:.0f} log/window')

In [ ]:
lengths = [len(w['log_keys']) for w in windows]
plt.hist(lengths, bins=50, color='#59c')
plt.title('Distribusi panjang window (jumlah log)')
plt.xlabel('log per window'); plt.ylabel('frekuensi')
plt.axvline(avg_len, color='r', ls='--', label=f'rata2 {avg_len:.0f}')
plt.legend(); plt.show()

## Tahap 4 — Split train/test

LogBERT dilatih HANYA pada window normal. Test = sisa normal + semua anomali.

In [ ]:
import random
random.seed(C.SEED)

normal = [w for w in windows if not w['is_anomaly']]
anom = [w for w in windows if w['is_anomaly']]
random.shuffle(normal)
n_train = int(len(normal) * C.TRAIN_RATIO)
train = normal[:n_train]
test = normal[n_train:] + anom
random.shuffle(test)

pickle.dump(train, open(C.TRAIN_PKL, 'wb'))
pickle.dump(test, open(C.TEST_PKL, 'wb'))
print(f'Train (normal): {len(train):,} | Test: {len(test):,} ({sum(w["is_anomaly"] for w in test):,} anomali)')

## Tahap 5 — Training LogBERT (fine-tune DistilBERT, MLKP)

Setiap Log Key didaftarkan sebagai **satu token** di vocabulary, sehingga masking
1 token = masking 1 Log Key utuh (**MLKP**, sesuai LogBERT), bukan potongan subword.

⚠️ Tahap GPU. Set `SMOKE=True` di Setup untuk uji cepat tanpa training penuh.

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForMaskedLM, DataCollatorForLanguageModeling
# Catatan: TIDAK memakai transformers.Trainer karena Trainer meng-import
# `datasets`→`aiohttp` yang gagal (SSL Windows cert store rusak). Loop manual.

train_data = train[:200] if SMOKE else train
print(f'Melatih pada {len(train_data):,} window' + (' [SMOKE]' if SMOKE else ''))

tok = AutoTokenizer.from_pretrained(C.BASE_MODEL)

# MLKP: daftarkan tiap Log Key unik sebagai 1 token → masking 1 token = 1 Log Key utuh
unique_keys = sorted({k for w in train_data for k in w['log_keys']})
tok.add_tokens([f'{C.LOGKEY_PREFIX}{k}' for k in unique_keys])
print(f'{len(unique_keys)} Log Key unik ditambahkan ke vocabulary')

def seq2txt(w):
    return ' '.join(f'{C.LOGKEY_PREFIX}{k}' for k in w['log_keys'][:C.MAX_LOG_KEYS])

class LogWindowDataset(torch.utils.data.Dataset):
    def __init__(self, windows, tok, max_length):
        self.enc = tok([seq2txt(w) for w in windows], truncation=True,
                       padding='max_length', max_length=max_length)
    def __len__(self):
        return len(self.enc['input_ids'])
    def __getitem__(self, i):
        return {k: torch.tensor(v[i]) for k, v in self.enc.items()}

collator = DataCollatorForLanguageModeling(tok, mlm=True, mlm_probability=C.MLM_PROBABILITY)
loader = DataLoader(LogWindowDataset(train_data, tok, C.MAX_LENGTH),
                    batch_size=C.BATCH_SIZE, shuffle=True, collate_fn=collator)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = AutoModelForMaskedLM.from_pretrained(C.BASE_MODEL)
model.resize_token_embeddings(len(tok))   # akomodasi token Log Key baru
model.to(dev).train()

opt = torch.optim.AdamW(model.parameters(), lr=5e-5)
fp16 = C.FP16 and dev == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=fp16)
epochs = 1 if SMOKE else C.EPOCHS

for epoch in range(epochs):
    running, n = 0.0, 0
    opt.zero_grad()
    for step, batch in enumerate(loader):
        batch = {k: v.to(dev) for k, v in batch.items()}
        with torch.amp.autocast('cuda', enabled=fp16):
            loss = model(**batch).loss / C.GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step + 1) % C.GRAD_ACCUM == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad()
        running += loss.item() * C.GRAD_ACCUM; n += 1
        if step % 50 == 0:
            print(f'  epoch {epoch+1} step {step}/{len(loader)} loss {loss.item()*C.GRAD_ACCUM:.4f}')
    print(f'  epoch {epoch+1} selesai | avg loss {running/max(n,1):.4f}')

model.save_pretrained(C.LOGBERT_DIR); tok.save_pretrained(C.LOGBERT_DIR)
print(f'LogBERT → {C.LOGBERT_DIR}')

## Tahap 6 — Training DeepSVDD

In [ ]:
import numpy as np
from transformers import AutoModel
from src.model.svdd import train_svdd   # DeepSVDD PyTorch murni (tanpa pyod/numba)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
bert = AutoModel.from_pretrained(C.LOGBERT_DIR).eval().to(dev)

def embed(ws, batch=64):
    out = []
    for i in range(0, len(ws), batch):
        texts = [seq2txt(w) for w in ws[i:i+batch]]   # format MLKP yang sama
        enc = tok(texts, return_tensors='pt', truncation=True, padding=True,
                  max_length=C.MAX_LENGTH).to(dev)
        with torch.no_grad():
            out.append(bert(**enc).last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(out)

X = embed(train_data)
print('Embedding:', X.shape)

net, center, scores = train_svdd(X, epochs=1 if SMOKE else C.SVDD_EPOCHS, device=dev)

torch.save({
    'state_dict': net.state_dict(),
    'center': center,                 # tensor CPU
    'in_dim': X.shape[1],
    'score_min': float(scores.min()),
    'score_max': float(scores.max()),
}, C.SVDD_MODEL)
print(f'DeepSVDD → {C.SVDD_MODEL} | rentang skor [{scores.min():.4f}, {scores.max():.4f}]')

## Selesai

Artefak: `models/logbert/`, `models/svdd.pt`, `models/drain3_state.bin`.

**Langkah berikutnya:** `src/model/predict.py` sudah siap memuat artefak ini.
Ganti impor di `scripts/verify_pipeline.py`:
`from src.model.stub import ...` → `from src.model.predict import predict`.